# Module 3- Missing Value Handling

## 1. What are Missing Values?

Missing values are values that are not available or not recorded for a particular observation in a dataset.

They can occur when information is not collected, is unavailable, or is left blank during data collection.

In Pandas, missing values are commonly represented as:

- `NaN` – Not a Number
- `None` – Python null value
- `NaT` – Not a Time, used for missing date/time values

Missing values should be identified and handled appropriately before performing data analysis or Machine Learning.

The appropriate handling technique depends on the type and amount of missing data.

In [2]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r"C:\Users\HP\sprint-5-data-cleaning-preprocessing\data\hotel_bookings.csv")

# Display the first five records
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
# Check for missing values in the dataset

missing_values = df.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values)

Missing Values in Each Column:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                     

### Observation

The number of missing values in each column was identified.

This helps us understand which columns contain missing data and provides a basis for selecting an appropriate missing value handling technique.

## 2. Why Missing Values Occur

Missing values can occur for several reasons during data collection, storage, or data entry.

Common reasons include:

- **Not provided:** The information was not given by the user or customer.
- **Not applicable:** The value does not apply to a particular record.
- **Data entry errors:** Information may be skipped or entered incorrectly.
- **System or technical issues:** Data may be lost during data collection or transfer.
- **Incomplete data collection:** Some information may not have been collected.
- **Data integration:** Different datasets may have missing fields when they are combined.

Understanding why values are missing is important because the reason for missingness helps determine the appropriate method for handling them.

In [4]:
# Check the number of missing values in each column

missing_values = df.isnull().sum()

print("Columns with Missing Values:")
print(missing_values[missing_values > 0])

Columns with Missing Values:
children         4
country        488
agent        16340
company     112593
dtype: int64


### Observation

The columns containing missing values were identified.

Understanding the possible reasons for missing values helps us choose an appropriate method for handling them instead of applying the same technique to every column.

## 3. MCAR (Missing Completely At Random)

MCAR means that the probability of a value being missing is completely independent of both the observed and unobserved data.

In simple terms, the missingness occurs randomly and is not related to any other variable in the dataset.

Example:

If some customer records are missing their age values because of a random system error, and the missingness is unrelated to the customer's age or any other information, the data can be considered MCAR.

### When to Use

- When missing values occur randomly.
- When the missingness is not related to any variable in the dataset.
- When removing or imputing the missing values is unlikely to introduce systematic bias.

### When Not to Use

- When missingness is related to another observed variable.
- When missingness depends on the missing value itself.
- When there is a clear pattern in the missing values.

### Advantages

- Missing values can often be handled using simple methods.
- Dropping affected rows may be reasonable when the amount of missing data is small.
- Less risk of systematic bias compared with MAR or MNAR.

### Limitations

- True MCAR is relatively uncommon in real-world datasets.
- Removing many rows can reduce the amount of available data.
- Assuming data is MCAR without investigation can lead to incorrect preprocessing decisions.

In [5]:
# Check the pattern of missing values

missing_counts = df.isnull().sum()

print("Missing Values:")
print(missing_counts[missing_counts > 0])

Missing Values:
children         4
country        488
agent        16340
company     112593
dtype: int64


### Observation

The missing value counts were examined to understand where missing data exists.

The presence of missing values alone does not prove that the data is MCAR. The reason and pattern of missingness should be investigated before assuming MCAR.

## 4. MAR (Missing At Random)

MAR means that the probability of a value being missing depends on other observed variables in the dataset, but not on the missing value itself.

In simple terms, the missingness can be explained by information that is already available in other columns.

Example:

If the income value is more likely to be missing for customers from a particular city, and the city information is available, the missingness may be considered MAR.

### When to Use

- When missingness is related to other observed variables.
- When the observed variables can help predict the missing values.
- When there is a clear relationship between missingness and other available features.

### When Not to Use

- When missingness is completely unrelated to the observed data.
- When the missingness depends directly on the missing value itself.
- When there is insufficient information to identify the variables related to missingness.

### Advantages

- Observed variables can be used to make better imputations.
- More information can be retained compared with simply dropping rows.
- Suitable for advanced imputation techniques.

### Limitations

- The MAR assumption cannot be directly verified from the observed data alone.
- Incorrectly assuming MAR can lead to biased results.
- Imputation may require additional analysis and more complex techniques.

In [6]:
# Compare missing values with another observed variable

missing_columns = df.columns[df.isnull().any()]

print("Columns with Missing Values:")
print(missing_columns.tolist())

for col in missing_columns:
    print(f"\nMissing values by hotel type for '{col}':")
    print(df.groupby('hotel')[col].apply(lambda x: x.isnull().sum()))

Columns with Missing Values:
['children', 'country', 'agent', 'company']

Missing values by hotel type for 'children':
hotel
City Hotel      4
Resort Hotel    0
Name: children, dtype: int64

Missing values by hotel type for 'country':
hotel
City Hotel       24
Resort Hotel    464
Name: country, dtype: int64

Missing values by hotel type for 'agent':
hotel
City Hotel      8131
Resort Hotel    8209
Name: agent, dtype: int64

Missing values by hotel type for 'company':
hotel
City Hotel      75641
Resort Hotel    36952
Name: company, dtype: int64


### Observation

The distribution of missing values was compared across an observed variable.

If the missingness differs across groups, it may indicate that the missingness is related to an observed variable. However, this analysis alone does not prove that the data is MAR.

## 5. MNAR (Missing Not At Random)

MNAR means that the probability of a value being missing depends on the value itself or on information that is not observed in the dataset.

In simple terms, the reason why a value is missing is related to the missing value itself.

Example:

If customers with very high incomes are more likely to leave their income information blank, the missingness may be considered MNAR because the missingness is related to the income value itself.

### When to Use

- When missingness appears to depend on the missing value itself.
- When there is a meaningful reason or domain knowledge supporting the missingness pattern.
- When the missing data mechanism needs to be considered during analysis.

### When Not to Use

- When missingness is completely random.
- When missingness can be explained by observed variables.
- When there is no evidence or domain knowledge suggesting MNAR.

### Advantages

- Helps recognize potentially systematic missingness.
- Prevents inappropriate use of simple imputation methods.
- Encourages the use of domain knowledge when handling missing data.

### Limitations

- MNAR cannot be determined from the observed data alone.
- It usually requires domain knowledge or additional information.
- Incorrect assumptions about MNAR can lead to biased results.

# Check whether missing values may show a pattern

missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing_Count': missing_values,
    'Missing_Percentage': missing_percentage
})

print("Missing Value Summary:")
print(missing_summary[missing_summary['Missing_Count'] > 0])

### Observation

The missing value pattern was examined across the dataset.

The presence of missing values alone does not confirm that the data is MNAR. Additional domain knowledge or information about why the values are missing is required to support an MNAR assumption.

## 6. Detecting Missing Values

Before handling missing values, we first need to identify which columns contain missing data and how many values are missing.

Pandas provides methods such as `isnull()` and `isna()` to detect missing values.

These methods return `True` when a value is missing and `False` when a value is present.

Detecting missing values helps us understand the extent of missing data before selecting an appropriate handling technique.

In [7]:
# Detect missing values in each column

missing_values = df.isnull().sum()

print("Missing Values in Each Column:")
print(missing_values[missing_values > 0])

Missing Values in Each Column:
children         4
country        488
agent        16340
company     112593
dtype: int64


### Observation

The columns containing missing values were identified along with their missing value counts.

This information will help us decide whether to drop, impute, or otherwise handle the missing values in the following steps.

## 7. Missing Value Percentage

Missing value percentage shows the proportion of missing values in each column compared with the total number of records.

It helps us understand the severity of missing data and decide which handling technique is appropriate.

A commonly used formula is:

Missing Value Percentage = (Number of Missing Values / Total Number of Values) × 100

A column with a very small percentage of missing values may be handled differently from a column with a large percentage of missing values.

In [8]:
# Calculate missing value percentage for each column

missing_count = df.isnull().sum()
missing_percentage = (missing_count / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing Percentage': missing_percentage
})

print("Missing Value Summary:")
print(missing_summary[missing_summary['Missing Count'] > 0])

Missing Value Summary:
          Missing Count  Missing Percentage
children              4            0.003350
country             488            0.408744
agent             16340           13.686238
company          112593           94.306893


### Observation

The missing value count and percentage were calculated for each column containing missing data.

The missing percentage helps determine the extent of missing data and supports the selection of a suitable handling technique.

## 8. Dropping Rows

Dropping rows means removing records that contain missing values from the dataset.

This approach can be useful when only a small number of rows contain missing values and removing them will not significantly reduce the dataset.

### When to Use

- When only a small percentage of rows contain missing values.
- When the missing records are not important for the analysis.
- When enough data remains after removing the rows.

### When Not to Use

- When a large number of rows contain missing values.
- When removing rows would result in significant data loss.
- When the missing values contain important information.

### Advantages

- Simple and easy to apply.
- Does not introduce estimated values.
- Keeps the remaining data unchanged.

### Limitations

- Can reduce the size of the dataset.
- May remove useful information.
- Can introduce bias if the missing values are not randomly distributed.

In [9]:
# Create a copy and drop rows containing missing values

df_drop_rows = df.dropna()

print("Original number of rows:", len(df))
print("Rows after dropping missing values:", len(df_drop_rows))
print("Rows removed:", len(df) - len(df_drop_rows))

Original number of rows: 119390
Rows after dropping missing values: 217
Rows removed: 119173


### Observation

Rows containing missing values were removed from a separate copy of the dataset.

The original `df` was not modified, allowing us to compare the number of records before and after row removal.

## 9. Dropping Columns

Dropping columns means removing an entire column that contains missing values from the dataset.

This approach can be considered when a column has a very high percentage of missing values and does not provide enough useful information for the analysis.

### When to Use

- When a column contains a very high percentage of missing values.
- When the column is not important for the analysis.
- When keeping the column provides little useful information.

### When Not to Use

- When the column contains important information.
- When only a small number of values are missing.
- When removing the column may affect the analysis or prediction.

### Advantages

- Simple and easy to apply.
- Removes the missing values completely.
- Avoids introducing estimated values.

### Limitations

- Important information may be lost.
- Can reduce the number of useful features.
- Removing an important feature may negatively affect Machine Learning performance.

In [10]:
# Identify columns with missing values

missing_percentage = (df.isnull().sum() / len(df)) * 100

print("Missing Value Percentage:")
print(missing_percentage[missing_percentage > 0])

# Create a copy and drop columns containing missing values
df_drop_columns = df.dropna(axis=1)

print("\nOriginal number of columns:", df.shape[1])
print("Columns after dropping:", df_drop_columns.shape[1])

Missing Value Percentage:
children     0.003350
country      0.408744
agent       13.686238
company     94.306893
dtype: float64

Original number of columns: 32
Columns after dropping: 28


### Observation

Columns containing missing values were identified, and a separate copy was created by removing columns with missing values.

The original dataset was not modified. In practice, columns should only be removed after considering their importance and the percentage of missing values.

## 10. Mean Imputation

Mean imputation replaces missing numerical values with the mean (average) of the available values in that column.

Mean is calculated as:

Mean = Sum of all available values / Number of available values

### When to Use

- When the column contains numerical data.
- When the data is approximately normally distributed.
- When the column does not contain many extreme outliers.
- When the percentage of missing values is relatively small.

### When Not to Use

- When the data contains significant outliers.
- When the data is highly skewed.
- When the missing percentage is very high.
- For categorical columns.

### Advantages

- Simple and easy to implement.
- Preserves the number of observations.
- Fast for large datasets.

### Limitations

- Can reduce the natural variability of the data.
- Can distort the distribution.
- Sensitive to outliers.
- May produce unrealistic values when the data is highly skewed.

In [11]:
# Example of Mean Imputation

df_mean = df.copy()

numeric_column = 'children'

mean_value = df_mean[numeric_column].mean()

df_mean[numeric_column] = df_mean[numeric_column].fillna(mean_value)

print("Mean value used for imputation:", mean_value)
print("\nMissing values after mean imputation:")
print(df_mean[numeric_column].isnull().sum())

Mean value used for imputation: 0.10388990333874994

Missing values after mean imputation:
0


### Observation

Missing values in the selected numerical column were replaced with the column mean.

Mean imputation is suitable only when the distribution and missing value pattern support its use. It should not be applied blindly to every numerical column.

## 11. Median Imputation

Median imputation replaces missing numerical values with the median of the available values in that column.

The median is the middle value when the observations are arranged in ascending or descending order.

### When to Use

- When the column contains numerical data.
- When the data is skewed.
- When the column contains outliers.
- When the percentage of missing values is relatively small.

### When Not to Use

- For categorical columns.
- When the missing percentage is very high.
- When the median does not represent the underlying data appropriately.

### Advantages

- Less affected by outliers than mean imputation.
- Simple and easy to implement.
- Preserves the number of observations.

### Limitations

- Can reduce the natural variability of the data.
- May distort the original distribution.
- Does not consider relationships between different columns.

In [12]:
# Example of Median Imputation

df_median = df.copy()

numeric_column = 'children'

median_value = df_median[numeric_column].median()

df_median[numeric_column] = df_median[numeric_column].fillna(median_value)

print("Median value used for imputation:", median_value)
print("\nMissing values after median imputation:")
print(df_median[numeric_column].isnull().sum())

Median value used for imputation: 0.0

Missing values after median imputation:
0


### Observation

Missing values in the selected numerical column were replaced with the column median.

Median imputation is particularly useful when numerical data is skewed or contains outliers because the median is less sensitive to extreme values.

## 12. Mode Imputation

Mode imputation replaces missing values with the most frequently occurring value in a column.

The mode is the value that appears most often in a dataset.

Mode imputation is mainly used for categorical data, but it can also be used for discrete numerical data when appropriate.

### When to Use

- When the column contains categorical data.
- When the most frequent category is a reasonable replacement.
- When the percentage of missing values is relatively small.

### When Not to Use

- When there are many missing values.
- When the categories have similar frequencies.
- When the mode does not represent the missing observations appropriately.

### Advantages

- Simple and easy to implement.
- Suitable for categorical data.
- Does not create a new category.
- Preserves the number of observations.

### Limitations

- Can increase the frequency of the most common category.
- May introduce bias.
- Does not consider relationships between other variables.
- Can reduce the natural variation in categorical data.

In [13]:
# Example of Mode Imputation

df_mode = df.copy()

categorical_column = 'meal'

mode_value = df_mode[categorical_column].mode()[0]

df_mode[categorical_column] = df_mode[categorical_column].fillna(mode_value)

print("Mode value used for imputation:", mode_value)
print("\nMissing values after mode imputation:")
print(df_mode[categorical_column].isnull().sum())

Mode value used for imputation: BB

Missing values after mode imputation:
0


### Observation

Missing values in the selected categorical column were replaced with its most frequently occurring value.

Mode imputation is appropriate for categorical data when the most frequent category is a reasonable representation of the missing values.

## 13. Constant Value Imputation

Constant value imputation replaces missing values with a predefined constant value.

For categorical columns, values such as `"Unknown"` or `"Not Available"` can be used.

For numerical columns, a meaningful constant such as `0` can be used when zero has a valid business meaning.

### When to Use

- When a missing value has a meaningful default value.
- When missingness itself carries useful information.
- For categorical columns where `"Unknown"` can represent missing information.
- For numerical columns when a specific constant has a valid meaning.

### When Not to Use

- When the chosen constant does not have a meaningful interpretation.
- When replacing missing values with zero would change the meaning of the data.
- When the missing percentage is very high without understanding the reason for missingness.

### Advantages

- Simple and easy to implement.
- Preserves the number of observations.
- Can make missingness explicitly identifiable.
- Useful for categorical data with an `"Unknown"` category.

### Limitations

- An inappropriate constant can introduce bias.
- May distort the distribution of the data.
- Does not use relationships between other variables.

In [14]:
# Example of Constant Value Imputation

df_constant = df.copy()

categorical_column = 'meal'

df_constant[categorical_column] = df_constant[categorical_column].fillna('Unknown')

print("Missing values after constant value imputation:")
print(df_constant[categorical_column].isnull().sum())

print("\nValue counts:")
print(df_constant[categorical_column].value_counts(dropna=False))

Missing values after constant value imputation:
0

Value counts:
meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64


### Observation

Missing values in the selected categorical column were replaced with the constant value `"Unknown"`.

This approach keeps the missing information identifiable instead of replacing it with an existing category.

## 14. Forward Fill

Forward fill replaces a missing value with the most recent non-missing value that appears before it.

In Pandas, forward fill can be performed using `ffill()`.

### When to Use

- When the order of observations is meaningful.
- For time-series or sequential data.
- When the previous value is a reasonable representation of the missing value.

### When Not to Use

- When the rows are randomly ordered.
- When consecutive values can change significantly.
- When there is no meaningful relationship between neighboring observations.

### Advantages

- Simple and easy to apply.
- Preserves the number of observations.
- Useful for sequential and time-series data.

### Limitations

- Can propagate an incorrect value across several missing records.
- May create long runs of the same value.
- Not suitable for all types of data.

In [15]:
# Example of Forward Fill

df_forward = df.copy()

column = 'children'

print("Missing values before forward fill:", df_forward[column].isnull().sum())

df_forward[column] = df_forward[column].ffill()

print("Missing values after forward fill:", df_forward[column].isnull().sum())

Missing values before forward fill: 4
Missing values after forward fill: 0


### Observation

Forward fill was applied to the selected column.

Missing values were replaced using the most recent available value before each missing value. This method is more appropriate when the order of observations has a meaningful relationship.

## 15. Backward Fill

Backward fill replaces a missing value with the next available non-missing value that appears after it.

In Pandas, backward fill can be performed using `bfill()`.

### When to Use

- When the order of observations is meaningful.
- For time-series or sequential data.
- When the next available value is a reasonable replacement for the missing value.

### When Not to Use

- When the rows are randomly ordered.
- When neighboring values can change significantly.
- When there is no meaningful relationship between consecutive observations.

### Advantages

- Simple and easy to apply.
- Preserves the number of observations.
- Useful for sequential and time-series data.

### Limitations

- Can propagate an incorrect value backward.
- May create repeated values.
- The missing values at the end of the dataset may remain missing if no later value is available.

In [16]:
# Example of Backward Fill

df_backward = df.copy()

column = 'children'

print("Missing values before backward fill:", df_backward[column].isnull().sum())

df_backward[column] = df_backward[column].bfill()

print("Missing values after backward fill:", df_backward[column].isnull().sum())

Missing values before backward fill: 4
Missing values after backward fill: 0


### Observation

Backward fill was applied to the selected column.

Missing values were replaced using the next available non-missing value. This technique is useful when the order of observations is meaningful and the following value is a suitable replacement.

## 16. Interpolation

Interpolation estimates missing numerical values using the values that are available around them.

Instead of replacing every missing value with the same value, interpolation uses the pattern or trend between known observations.

A common method is linear interpolation.

### When to Use

- When the data is numerical.
- When observations have a meaningful order.
- For time-series or sequential data.
- When the values are expected to change gradually.

### When Not to Use

- For categorical data.
- When observations are not ordered.
- When the data changes irregularly or has large gaps.
- When interpolation does not have a meaningful interpretation for the variable.

### Advantages

- Uses surrounding observations.
- Can preserve the trend of the data.
- Usually provides more realistic values than a constant replacement.

### Limitations

- Estimated values are not actual observations.
- Results depend on the selected interpolation method.
- Can be inaccurate when there are large gaps or sudden changes.

In [17]:
# Example of Linear Interpolation

df_interpolation = df.copy()

column = 'children'

print("Missing values before interpolation:", df_interpolation[column].isnull().sum())

df_interpolation[column] = df_interpolation[column].interpolate(method='linear')

print("Missing values after interpolation:", df_interpolation[column].isnull().sum())

Missing values before interpolation: 4
Missing values after interpolation: 0


### Observation

Linear interpolation was applied to the selected numerical column.

The missing values were estimated using the values surrounding them. This method is useful when the data has a meaningful order and follows a relatively continuous pattern.

## 17. Group-Based Imputation

Group-based imputation replaces missing values using a statistic calculated within a relevant group.

For example, missing numerical values can be replaced with the median or mean of the corresponding group instead of using one overall value for the entire dataset.

In the hotel bookings dataset, groups such as `hotel` can be used to calculate separate values.

### When to Use

- When the data contains meaningful groups.
- When different groups have different distributions.
- When the group variable is relevant to the missing column.

### When Not to Use

- When there is no meaningful grouping variable.
- When some groups contain very few observations.
- When the group statistics are unreliable.

### Advantages

- Uses information specific to each group.
- Can provide more realistic values than overall mean or median imputation.
- Preserves differences between groups.

### Limitations

- Results depend on the quality of the selected grouping variable.
- Small groups may produce unreliable estimates.
- More complex than simple mean or median imputation.

In [18]:
# Example of Group-Based Median Imputation

df_group = df.copy()

column = 'children'
group_column = 'hotel'

df_group[column] = df_group.groupby(group_column)[column].transform(
    lambda x: x.fillna(x.median())
)

print("Missing values after group-based imputation:")
print(df_group[column].isnull().sum())

Missing values after group-based imputation:
0


### Observation

Missing values were imputed using the median calculated separately within each hotel group.

This approach considers group-level differences and can be more appropriate than using a single overall median.

## 18. KNN Imputation

KNN (K-Nearest Neighbors) imputation replaces missing values using values from similar observations.

It identifies the nearest observations based on other numerical features and uses their values to estimate the missing value.

### When to Use

- When numerical features have meaningful relationships.
- When similar observations are expected to have similar values.
- When enough relevant numerical features are available.
- When a more data-driven imputation method is required.

### When Not to Use

- When the dataset is very large, because KNN can be computationally expensive.
- When numerical features are on very different scales without proper scaling.
- When there are not enough useful features to identify similar observations.
- For purely categorical data.

### Advantages

- Uses relationships between observations.
- Can provide more realistic estimates than simple mean or median imputation.
- Considers multiple features when finding similar observations.

### Limitations

- Computationally expensive for large datasets.
- Sensitive to feature scaling.
- Choice of the number of neighbors (`k`) affects the results.
- Can produce inaccurate values when similar observations are not available.

In [ ]:
from sklearn.impute import KNNImputer

df_knn = df.copy()

# Select numerical columns
knn_columns = ['children', 'babies', 'adults']

# Apply KNN imputation
imputer = KNNImputer(n_neighbors=5)

df_knn[knn_columns] = imputer.fit_transform(df_knn[knn_columns])

print("Missing values after KNN Imputation:")
print(df_knn[knn_columns].isnull().sum())

### Observation

KNN imputation was applied to the numerical columns using the five nearest observations.

The missing values were estimated based on similar records rather than using a single overall statistic such as the mean or median.

## 19. Iterative Imputation

Iterative imputation estimates missing values by modeling each feature with missing values as a function of other features.

The process works iteratively:

1. Initial values are assigned to the missing entries.
2. One feature with missing values is selected as the target.
3. Other available features are used to predict the missing values.
4. The estimated values are used to update the dataset.
5. The process is repeated for multiple iterations.

### When to Use

- When numerical features have meaningful relationships with each other.
- When simple imputation methods may not provide accurate estimates.
- When the dataset contains multiple numerical features with missing values.

### When Not to Use

- When the dataset is very small.
- When numerical features have weak relationships.
- When computational simplicity is required.
- When the assumptions of the underlying model are not appropriate.

### Advantages

- Uses relationships between multiple features.
- Can provide better estimates than simple mean or median imputation.
- Can handle missing values across multiple numerical columns.

### Limitations

- More computationally expensive than simple imputation.
- Results depend on the underlying model and selected parameters.
- Can be affected by outliers and noisy data.
- More complex to understand and implement.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df_iterative = df.copy()

# Select numerical columns
iterative_columns = ['children', 'babies', 'adults']

# Apply Iterative Imputation
imputer = IterativeImputer(max_iter=10, random_state=42)

df_iterative[iterative_columns] = imputer.fit_transform(
    df_iterative[iterative_columns]
)

print("Missing values after Iterative Imputation:")
print(df_iterative[iterative_columns].isnull().sum())

### Observation

Iterative imputation was applied to the numerical columns.

The missing values were estimated using relationships between the available numerical features over multiple iterations.

This method is more advanced than simple mean or median imputation because it considers information from other features.